In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
import joblib

In [2]:
DATASET_PATH = "../dataset/bccc-cpacket-cloud-ddos-2024-merged.parquet"

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at: {DATASET_PATH}")

try:
    df = pd.read_parquet(DATASET_PATH, engine="pyarrow")
except Exception as e:
    print(f"[Warn] read_parquet(engine='pyarrow') failed: {e}")
    df = pd.read_parquet(DATASET_PATH)

df.head()

,src_port,dst_port,duration,packets_count,fwd_packets_count,bwd_packets_count,total_payload_bytes,fwd_total_payload_bytes,bwd_total_payload_bytes,payload_bytes_max,...,max_fwd_payload_bytes_delta_len,mean_fwd_payload_bytes_delta_len,mode_fwd_payload_bytes_delta_len,variance_fwd_payload_bytes_delta_len,std_fwd_payload_bytes_delta_len,median_fwd_payload_bytes_delta_len,skewness_fwd_payload_bytes_delta_len,cov_fwd_payload_bytes_delta_len,label,activity
0,54573,25094,0.000063,3,2,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
1,25094,54573,0.000000,1,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
2,54573,25094,0.000028,3,1,2,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
3,9147,18060,0.000055,3,2,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
4,18060,9147,0.000000,1,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign


In [3]:
# Drop unnecessary 'activity' column
if "activity" in df.columns:
    df = df.drop(columns=["activity"])
    
# Label Encode the target variable
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

print("Label mapping (class -> encoded):")
for cls in le.classes_:
    print(f"  {cls}: {int(le.transform([cls])[0])}")

Label mapping (class -> encoded):
  Attack: 0
  Benign: 1
  Suspicious: 2


In [4]:
X = df.drop(columns=["label"])
y = df["label"]

# First split: 80% Train, 20% Temp (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: Split Temp into 50% Val, 50% Test (which means 10% / 10% of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Shapes:")
print("  X_train:", X_train.shape, "y_train:", y_train.shape)
print("  X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("  X_test: ", X_test.shape, "y_test: ", y_test.shape)

Shapes:
  X_train: (432395, 317) y_train: (432395,)
  X_val:   (54049, 317) y_val:   (54049,)
  X_test:  (54050, 317) y_test:  (54050,)


In [5]:
# 1) Remove zero-variance features (based on train)
zero_var_cols = [col for col in X_train.columns if X_train[col].nunique(dropna=False) <= 1]
if zero_var_cols:
    X_train = X_train.drop(columns=zero_var_cols)
    X_val   = X_val.drop(columns=zero_var_cols)
    X_test  = X_test.drop(columns=zero_var_cols)
print(f"[Prep] Dropped {len(zero_var_cols)} zero-variance columns.")

# 2) Remove highly correlated features (>0.95) using a sample of train to prevent multicollinearity
sample_n = min(50_000, len(X_train))
corr_sample = X_train.sample(n=sample_n, random_state=42)
corr_matrix = corr_sample.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
if high_corr_cols:
    X_train = X_train.drop(columns=high_corr_cols)
    X_val   = X_val.drop(columns=high_corr_cols)
    X_test  = X_test.drop(columns=high_corr_cols)
print(f"[Prep] Dropped {len(high_corr_cols)} highly-correlated columns (>0.95).")

print(f"Final Feature Space: {X_train.shape[1]} columns")

[Prep] Dropped 3 zero-variance columns.
[Prep] Dropped 152 highly-correlated columns (>0.95).
Final Feature Space: 162 columns


In [8]:
import joblib

preprocess_info = {
    "zero_var_cols": zero_var_cols,
    "high_corr_cols": high_corr_cols,
    "feature_columns": X_train.columns.tolist(),
}

joblib.dump(preprocess_info, "logreg_preprocess_info.joblib")
print("[Output] Preprocessing info saved.")

[Output] Preprocessing info saved.


In [ ]:
# Logistic Regression Pipeline
# 1. SimpleImputer: Fill missing values with median
# 2. StandardScaler: Standardize features to mean=0, variance=1
# 3. LogisticRegression: The model

logreg_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                class_weight="balanced", # Handle class imbalance
                max_iter=5000,           # Higher iter to ensure convergence
                solver="lbfgs",          # Default robust solver
                random_state=42,
                n_jobs=-1                # Use all CPU cores
            ),
        ),
    ]
)

print("[LogReg] Fitting...")
logreg_clf.fit(X_train, y_train)
print("[LogReg] Fit complete.")

val_pred = logreg_clf.predict(X_val)
print(f"[Val] Macro F1:    {f1_score(y_val, val_pred, average='macro'):.4f}")
print(f"[Val] Weighted F1: {f1_score(y_val, val_pred, average='weighted'):.4f}")

In [ ]:
print("[LogReg] Predicting on test set...")
y_pred_encoded = logreg_clf.predict(X_test)

y_pred_labels = le.inverse_transform(y_pred_encoded)
y_test_labels = le.inverse_transform(y_test)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test_labels, y_pred_labels, digits=4))

macro_f1 = f1_score(y_test_labels, y_pred_labels, average="macro")
weighted_f1 = f1_score(y_test_labels, y_pred_labels, average="weighted")
print("KEY METRICS:")
print(f"  Macro F1:    {macro_f1:.4f}  <- Primary")
print(f"  Weighted F1: {weighted_f1:.4f}  <- Secondary")

In [ ]:
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=le.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix - Logistic Regression", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("logreg_confusion_matrix.png", dpi=150)
plt.show()
print("[Output] Confusion matrix saved -> logreg_confusion_matrix.png")

In [ ]:
model_filename = 'logistic_regression_model.joblib'
joblib.dump(logreg_clf, model_filename)
print(f"[Output] Trained model saved to '{model_filename}'")

In [ ]:
# Testing the saved model with dummy data
import joblib
import pandas as pd
import numpy as np

# Load the pipeline
loaded_logreg = joblib.load('logistic_regression_model.joblib')

# Generate dummy data matching the features (X_train columns)
# Note: The pipeline handles scaling and imputation
dummy_data = pd.DataFrame(np.random.rand(1, X_train.shape[1]), columns=X_train.columns)

# Predict
prediction = loaded_logreg.predict(dummy_data)
print(f"Prediction for dummy data: {prediction}")